In [ ]:
# Cell 1 — Upload and inspect the DECRS file
from google.colab import files
import pandas as pd
import io

# Upload drls_reg.txt from your computer
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load as Excel file
df = pd.read_excel(
    io.BytesIO(uploaded[filename])
)

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nFirst row:\n{df.iloc[0]}")

Saving drls_reg.xls to drls_reg (1).xls


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Shape: (10460, 14)

Columns:
['FEI_NUMBER', 'DUNS_NUMBER', 'FIRM_NAME', 'ADDRESS', 'EXPIRATION_DATE', 'OPERATIONS', 'ESTABLISHMENT_CONTACT_NAME', 'ESTABLISHMENT_CONTACT_EMAIL', 'AGENT_DETAILS', 'REGISTRANT_NAME', 'REGISTRANT_DUNS', 'REGISTRANT_CONTACT_NAME', 'REGISTRANT_CONTACT_EMAIL', 'EXCLUSION_FLAG']

First row:
FEI_NUMBER                                                            0000000360
DUNS_NUMBER                                                            271408412
FIRM_NAME                                                                    DSP
ADDRESS                        Rue Grands Navoirs, Chauny,  F-02300, France (...
EXPIRATION_DATE                                                       12/31/2026
OPERATIONS                                                       API MANUFACTURE
ESTABLISHMENT_CONTACT_NAME                                         Sylvie Proisy
ESTABLISHMENT_CONTACT_EMAIL                             sylvie.proisy@dupont.com
AGENT_DETAILS                  1392

In [ ]:
# Cell 2 — Extract country codes from ADDRESS field
import re

def extract_country_code(address):
    if pd.isna(address):
        return 'UNKNOWN'
    match = re.search(r'\(([A-Z]{3})\)\s*$', str(address).strip())
    return match.group(1) if match else 'UNKNOWN'

df['COUNTRY_CODE'] = df['ADDRESS'].apply(extract_country_code)

# Sanity check
n_parsed = (df['COUNTRY_CODE'] != 'UNKNOWN').sum()
n_unknown = (df['COUNTRY_CODE'] == 'UNKNOWN').sum()

print(f"Total rows:       {len(df):,}")
print(f"Country parsed:   {n_parsed:,}")
print(f"Could not parse:  {n_unknown:,}")
print(f"Unique countries: {df['COUNTRY_CODE'].nunique()}")

# Show any rows that failed to parse
if n_unknown > 0:
    print("\nSample unparsed addresses:")
    print(df[df['COUNTRY_CODE'] == 'UNKNOWN']['ADDRESS'].head(5).to_string())

Total rows:       10,460
Country parsed:   10,460
Could not parse:  0
Unique countries: 82


In [ ]:
# Cell 3 — Geographic concentration analysis
country_counts = (
    df.groupby('COUNTRY_CODE')
    .size()
    .reset_index(name='FACILITY_COUNT')
    .sort_values('FACILITY_COUNT', ascending=False)
)

total = len(df)
country_counts['PCT_OF_TOTAL'] = (country_counts['FACILITY_COUNT'] / total * 100).round(2)

# Domestic vs foreign split — the headline number
domestic = country_counts.loc[country_counts['COUNTRY_CODE'] == 'USA', 'FACILITY_COUNT'].values[0]
foreign = total - domestic

print("=" * 50)
print(f"TOTAL ESTABLISHMENTS:    {total:,}")
print(f"DOMESTIC (USA):          {domestic:,}  ({domestic/total*100:.1f}%)")
print(f"FOREIGN:                 {foreign:,}  ({foreign/total*100:.1f}%)")
print("=" * 50)
print(f"\nTOP 15 COUNTRIES:")
print(country_counts.head(15).to_string(index=False))

TOTAL ESTABLISHMENTS:    10,460
DOMESTIC (USA):          5,659  (54.1%)
FOREIGN:                 4,801  (45.9%)

TOP 15 COUNTRIES:
COUNTRY_CODE  FACILITY_COUNT  PCT_OF_TOTAL
         USA            5659         54.10
         CHN            1178         11.26
         IND            1062         10.15
         DEU             257          2.46
         CAN             228          2.18
         FRA             190          1.82
         ITA             183          1.75
         KOR             159          1.52
         GBR             157          1.50
         JPN             152          1.45
         ESP             144          1.38
         CHE             112          1.07
         MEX              92          0.88
         IRL              81          0.77
         TWN              64          0.61


In [ ]:
# Cell 4 — API Manufacture concentration specifically
# Parse OPERATIONS (semicolon-delimited) into a clean list
df['OPS_LIST'] = df['OPERATIONS'].fillna('').apply(
    lambda x: [op.strip() for op in x.split(';') if op.strip()]
)

# Flag key operation types
df['IS_API']          = df['OPS_LIST'].apply(lambda ops: 'API MANUFACTURE' in ops)
df['IS_MANUFACTURE']  = df['OPS_LIST'].apply(lambda ops: 'MANUFACTURE' in ops)
df['IS_ANALYSIS']     = df['OPS_LIST'].apply(lambda ops: 'ANALYSIS' in ops)
df['IS_ANIMAL_FEED']  = df['OPS_LIST'].apply(lambda ops: 'MEDICATED ANIMAL FEED MANUFACTURE' in ops)

# API-only concentration
api_df = df[df['IS_API']]
api_foreign = api_df[api_df['COUNTRY_CODE'] != 'USA']

print(f"API MANUFACTURE facilities:         {len(api_df):,}")
print(f"  — Domestic (USA):                 {(api_df['COUNTRY_CODE'] == 'USA').sum():,}")
print(f"  — Foreign:                        {(api_df['COUNTRY_CODE'] != 'USA').sum():,}  ({len(api_foreign)/len(api_df)*100:.1f}%)")
print(f"\nTop foreign API manufacturers:")
print(
    api_foreign.groupby('COUNTRY_CODE')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='COUNT')
    .to_string(index=False)
)

API MANUFACTURE facilities:         2,509
  — Domestic (USA):                 531
  — Foreign:                        1,978  (78.8%)

Top foreign API manufacturers:
COUNTRY_CODE  COUNT
         IND    563
         CHN    532
         ITA     91
         DEU     89
         JPN     86
         FRA     65
         ESP     58
         GBR     46
         KOR     43
         CAN     43


In [ ]:
# Cell 5 — Map ISO codes to country names
country_name_map = {
    'USA': 'United States', 'CHN': 'China', 'IND': 'India',
    'DEU': 'Germany', 'CAN': 'Canada', 'FRA': 'France',
    'ITA': 'Italy', 'KOR': 'South Korea', 'GBR': 'United Kingdom',
    'JPN': 'Japan', 'ESP': 'Spain', 'CHE': 'Switzerland',
    'IRL': 'Ireland', 'TWN': 'Taiwan', 'BEL': 'Belgium',
    'NLD': 'Netherlands', 'AUS': 'Australia', 'ISR': 'Israel',
    'SGP': 'Singapore', 'MEX': 'Mexico', 'ARG': 'Argentina',
    'BRA': 'Brazil', 'ZAF': 'South Africa', 'POL': 'Poland',
    'SWE': 'Sweden', 'DNK': 'Denmark', 'AUT': 'Austria',
    'PRT': 'Portugal', 'HUN': 'Hungary', 'CZE': 'Czech Republic',
    'TUR': 'Turkey', 'PAK': 'Pakistan', 'BGD': 'Bangladesh',
    'NZL': 'New Zealand', 'FIN': 'Finland', 'NOR': 'Norway',
    'SVN': 'Slovenia', 'GRC': 'Greece', 'ROU': 'Romania',
    'BGR': 'Bulgaria', 'HRV': 'Croatia', 'PHL': 'Philippines',
    'THA': 'Thailand', 'MYS': 'Malaysia', 'VNM': 'Vietnam',
    'IDN': 'Indonesia', 'EGY': 'Egypt', 'JOR': 'Jordan',
    'LKA': 'Sri Lanka', 'ARE': 'United Arab Emirates',
    'SAU': 'Saudi Arabia', 'MAR': 'Morocco', 'NGA': 'Nigeria',
    'KEN': 'Kenya', 'TUN': 'Tunisia', 'CHL': 'Chile',
    'COL': 'Colombia', 'URY': 'Uruguay', 'PER': 'Peru',
    'ECU': 'Ecuador', 'GTM': 'Guatemala', 'CRI': 'Costa Rica',
    'CYP': 'Cyprus', 'MLT': 'Malta', 'LUX': 'Luxembourg',
    'EST': 'Estonia', 'LVA': 'Latvia', 'LTU': 'Lithuania',
    'SVK': 'Slovakia', 'MKD': 'North Macedonia', 'SRB': 'Serbia',
    'ALB': 'Albania', 'BIH': 'Bosnia', 'GEO': 'Georgia',
    'ARM': 'Armenia', 'AZE': 'Azerbaijan', 'KAZ': 'Kazakhstan',
    'IRN': 'Iran', 'IRQ': 'Iraq', 'LBN': 'Lebanon',
    'UKR': 'Ukraine', 'RUS': 'Russia', 'BLR': 'Belarus',
    'ETH': 'Ethiopia', 'GHA': 'Ghana', 'TZA': 'Tanzania',
    'CMR': 'Cameroon', 'ZWE': 'Zimbabwe', 'MOZ': 'Mozambique',
    'DZA': 'Algeria', 'SDN': 'Sudan', 'LBY': 'Libya',
    'SEN': 'Senegal', 'CIV': "Ivory Coast", 'UGA': 'Uganda',
}

df['COUNTRY_NAME'] = df['COUNTRY_CODE'].map(country_name_map).fillna(df['COUNTRY_CODE'])

In [ ]:
# Cell 6 — Build and export all CSVs for Tableau

# 1. Overall country summary
country_summary = (
    df.groupby(['COUNTRY_CODE', 'COUNTRY_NAME'])
    .size()
    .reset_index(name='FACILITY_COUNT')
    .sort_values('FACILITY_COUNT', ascending=False)
)
country_summary['PCT_OF_TOTAL'] = (country_summary['FACILITY_COUNT'] / len(df) * 100).round(2)
country_summary['IS_DOMESTIC'] = country_summary['COUNTRY_CODE'] == 'USA'
country_summary.to_csv('decrs_country_summary.csv', index=False)

# 2. API-only by country
api_country = (
    df[df['IS_API']]
    .groupby(['COUNTRY_CODE', 'COUNTRY_NAME'])
    .size()
    .reset_index(name='API_FACILITY_COUNT')
    .sort_values('API_FACILITY_COUNT', ascending=False)
)
api_country['PCT_OF_API_TOTAL'] = (api_country['API_FACILITY_COUNT'] / len(df[df['IS_API']]) * 100).round(2)
api_country.to_csv('decrs_api_country.csv', index=False)

# 3. Operation type breakdown by country (for stacked charts)
# Explode so each operation gets its own row
ops_exploded = df[['FEI_NUMBER', 'COUNTRY_CODE', 'COUNTRY_NAME', 'OPS_LIST']].explode('OPS_LIST')
ops_exploded = ops_exploded.rename(columns={'OPS_LIST': 'OPERATION_TYPE'})
ops_exploded = ops_exploded[ops_exploded['OPERATION_TYPE'].notna() & (ops_exploded['OPERATION_TYPE'] != '')]
ops_summary = (
    ops_exploded.groupby(['COUNTRY_CODE', 'COUNTRY_NAME', 'OPERATION_TYPE'])
    .size()
    .reset_index(name='COUNT')
)
ops_summary.to_csv('decrs_ops_by_country.csv', index=False)

# 4. Headline summary (for big-number cards)
headline = pd.DataFrame([
    {'METRIC': 'Total Establishments',         'VALUE': len(df),                      'CATEGORY': 'all'},
    {'METRIC': 'Domestic (USA)',                'VALUE': (df['COUNTRY_CODE']=='USA').sum(), 'CATEGORY': 'all'},
    {'METRIC': 'Foreign',                       'VALUE': (df['COUNTRY_CODE']!='USA').sum(), 'CATEGORY': 'all'},
    {'METRIC': 'API Manufacture - Total',       'VALUE': len(df[df['IS_API']]),            'CATEGORY': 'api'},
    {'METRIC': 'API Manufacture - Domestic',    'VALUE': (df[df['IS_API']]['COUNTRY_CODE']=='USA').sum(), 'CATEGORY': 'api'},
    {'METRIC': 'API Manufacture - Foreign',     'VALUE': (df[df['IS_API']]['COUNTRY_CODE']!='USA').sum(), 'CATEGORY': 'api'},
])
headline['PCT'] = headline.apply(
    lambda r: round(r['VALUE'] / len(df) * 100, 1) if r['CATEGORY'] == 'all'
    else round(r['VALUE'] / len(df[df['IS_API']]) * 100, 1), axis=1
)
headline.to_csv('decrs_headline.csv', index=False)

print("Exported:")
print(f"  decrs_country_summary.csv  — {len(country_summary)} countries")
print(f"  decrs_api_country.csv      — {len(api_country)} countries with API facilities")
print(f"  decrs_ops_by_country.csv   — {len(ops_summary)} operation-country combinations")
print(f"  decrs_headline.csv         — headline stats")

Exported:
  decrs_country_summary.csv  — 82 countries
  decrs_api_country.csv      — 56 countries with API facilities
  decrs_ops_by_country.csv   — 448 operation-country combinations
  decrs_headline.csv         — headline stats


In [ ]:
# Cell 7 — Download all files to your computer
from google.colab import files
files.download('decrs_country_summary.csv')
files.download('decrs_api_country.csv')
files.download('decrs_ops_by_country.csv')
files.download('decrs_headline.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>